<a href="https://colab.research.google.com/github/ThisumiWijesinghe/Evidence-Verified-RAG-Workflow-/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Ollama

In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 93 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,087 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Start Ollama Server in background

In [3]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for the server to start
time.sleep(5)
print("Ollama server started, PID:", process.pid)


Ollama server started, PID: 2641


#Pull LIama 3.1

In [4]:
!ollama pull llama3.1:8b

#Pull the embedding model

In [5]:
!ollama pull nomic-embed-text

# Verify both are installed

In [6]:
!ollama list

NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
llama3.1:8b                46e0c10c039e    4.9 GB    5 seconds ago             


#Test generation works

In [7]:
import requests
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.1:8b",
        "prompt": "Say 'setup successful' and nothing else.",
        "stream": False,
    }
)
print(response.json()["response"])

Setup successful.


#Install Langchain+dependencies for next steps

In [8]:
!pip install -q langchain langchain-community langchain-ollama faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Connect Langchain to Ollama

In [9]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.1:8b", temperature=0.2)
result = llm.invoke("In one sentence, what is Retrieval Augmented Generation?")
print(result)

Retrieval Augmented Generation (RAG) is a type of artificial intelligence model that combines the strengths of retrieval-based and generative models by first retrieving relevant information from a large database and then generating text based on that information.


#PDF Ingestion + Chunking

## Upload a test PDF to colab

In [10]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded:",pdf_path)

Saving dl based detecting malicious packages in pypl.pdf to dl based detecting malicious packages in pypl.pdf
Saving On the Performance of Malware Detection.pdf to On the Performance of Malware Detection.pdf
Saving Quantum redy crypto.pdf to Quantum redy crypto.pdf
Uploaded: dl based detecting malicious packages in pypl.pdf


#Install the text-splitter package

In [11]:
!pip install -q langchain langchain-community langchain-ollama  langchain-text-splitters faiss-cpu pypdf

##Load the PDF with PyPDFLoader

In [12]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages")
print("Sample text from page 1:\n", pages[0].page_content[:500])

/tmp/ipykernel_1444/2514516427.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 26 pages
Sample text from page 1:
 eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan University
Joondalup, WA, Australia
Abu Bakar Siddique Mahi
Research Graduate School, BUBT
Dhaka, Bangladesh
Gowri Ramachandran
Queensland University of Technology
Brisbane, QLD, Australia
Abstract
The s


##Chunk the text with RecursiveChracterTextSplitter

In [13]:
!pip install -q -U langchain-text-splitters


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("import successful!")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunks")
print("Sample text from chunk 1:\n", chunks[0].page_content[:300])

import successful!
Split into 148 chunks
Sample text from chunk 1:
 eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan Uni


##Quick sanity check -- Inspect a few chunks

In [15]:
for i in range(min(3, len(chunks))):
  print(f"---chunk {i} ---")
  print(chunks[i].page_content[:200])
  print("Metadata:", chunks[i].metadata)
  print()

---chunk 0 ---
eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
R
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '2026-04-30T00:20:09+00:00', 'author': 'Sk Tanzir Mehedi; Raja Jurdak; Chadni Islam; Abu Bakar Siddique Mahi; Gowri Ramachandran', 'doi': 'https://doi.org/10.48550/arXiv.2604.26219', 'keywords': 'Deep learning, malicious detection, software supply chain security, dynamic analysis, stability, explainable AI.', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'moddate': '2026-04-30T00:20:09+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem', 'trapped': '/False', 'arxivid': 

#Embeddings & FAISS vector store , test retrieval

##Connect the nomic-embed-text

In [16]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Embedding model connected succesfully!")

Embedding model connected succesfully!


##Test the embedding model

In [17]:
test_vector = embeddings.embed_query(
    "What is malicious package detection?"
)

print("Embedding dimension:", len(test_vector))
print("first 10 values:", test_vector[:10])

Embedding dimension: 768
first 10 values: [0.07384906, 0.020637663, -0.15453403, -0.018535705, 0.06982891, -0.04924275, -0.005812537, 0.03157499, -0.0047331247, -0.0016610859]


## Create the FAISS vector database

In [18]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS Vector database created successfully!")
print("Number of chunks stored:", len(chunks))

FAISS Vector database created successfully!
Number of chunks stored: 148


#FAISS Test retrieval

In [21]:

# ============================================================
# FIX: RESTART OLLAMA SERVER
# ============================================================

import subprocess
import time
import requests

# Start Ollama without blocking the notebook
ollama_log = open("/tmp/ollama_server.log", "a")

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT
)

# Wait until Ollama responds
server_ready = False

for attempt in range(20):
    try:
        response = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=3
        )

        if response.status_code == 200:
            server_ready = True
            break

    except requests.exceptions.RequestException:
        pass

    time.sleep(1)

if server_ready:
    print("Ollama server started successfully!")
    print("Available models:")
    print(requests.get(
        "http://127.0.0.1:11434/api/tags"
    ).json())
else:
    print("Ollama server failed to start.")
    print("Check /tmp/ollama_server.log")


Ollama server started successfully!
Available models:
{'models': [{'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'modified_at': '2026-10-17T18:19:28.984143837Z', 'size': 274302450, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16', 'context_length': 2048, 'embedding_length': 768, 'runner': 'ggml'}, 'capabilities': ['embedding']}, {'name': 'llama3.1:8b', 'model': 'llama3.1:8b', 'modified_at': '2026-10-17T18:19:23.531758448Z', 'size': 4920753328, 'digest': '46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 4096, 'runner': 'ggml'}, 'capabilities': ['completion', 'tools']}]}


In [22]:

# ============================================================
# TEST EMBEDDING MODEL
# ============================================================

import time

start = time.time()

test_embedding = embeddings.embed_query(
    "What problem does eDySec aim to solve?"
)

print("Embedding successful!")
print("Embedding dimension:", len(test_embedding))
print("Time:", round(time.time() - start, 2), "seconds")


Embedding successful!
Embedding dimension: 768
Time: 12.73 seconds


In [23]:

# ============================================================
# TEST FAISS RETRIEVAL
# ============================================================

question = "What problem does eDySec aim to solve?"

retrieved_docs = vectorstore.similarity_search(
    question,
    k=4
)

print("Retrieved chunks:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs):
    print(f"\n{'='*60}")
    print(f"Retrieved Chunk {i+1}")
    print("Page:", doc.metadata.get("page_label", "Unknown"))
    print("="*60)
    print(doc.page_content[:500])


Retrieved chunks: 4

Retrieved Chunk 1
Page: 6
Conference Name, Month Year, City, Country Tanzir et al.
between the selected feature subset and the model’s learned be-
havior is assessed throughC=F ∗∩Feat(R 𝑒𝑥𝑝), where a larger
overlap indicates stronger agreement between the selected features
and the features that most strongly influence model predictions.
Algorithm 1 presents the pseudo-code for the proposed eDySec
framework workflow.
4 eDySec Design and Setup
This section describes the experimental design of eDySec, including
the computat

Retrieved Chunk 2
Page: 13
eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem Conference Name, Month Year, City, Country
References
[1] Hubert Baniecki and Przemyslaw Biecek. 2024. Adversarial Attacks and Defenses
in Explainable Artificial Intelligence: A Survey.Information Fusion107 (2024),
102303. doi:10.1016/j.inffus.2024.102303
[2] Alessio Benavoli, Giorgio Corani, Janez Demš